# Chapter 14 · Two Layers, Three Neurons

### Hidden neurons move the points so one line can win.

*Part 3 · Neural networks*

---

This notebook is the same chapter as the app, but with the code showing.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from kidsml.datasets import toy_shape, xor_exact
from kidsml.nn_numpy import MLP
from kidsml.nnplots import boundary_with_hidden, hidden_surfaces_figure, mlp_snapshot_training, model_from_snapshot
from kidsml.plots import decision_boundary, loss_curve, use_house_style

use_house_style()

## 🎣 Start here

XOR is back because it is the test that tells us whether Part 3 worked.

One neuron cannot solve it: Chapter 3 proved one straight line cannot put opposite
corners together. The escape route was “invent better features.” A hidden layer does that
for us, then the output neuron runs the Chapter 2 line trick on those new features.

```mermaid
graph LR
    X1[x₁] --> H1[h₁]
    X1 --> H2[h₂]
    X2[x₂] --> H2
    X2 --> H3[h₃]
    H1 --> O[output neuron]
    H2 --> O
    H3 --> O
```

Read left to right: two original inputs feed three hidden neurons, and those three
reports feed one final neuron.

## ✏️ Work it out

We will make two hidden features by hand: **OR-ish** and **AND-ish**. This table is the
whole XOR story in miniature.

In the original `x1, x2` square, the red points sit in opposite corners. No straight line
can grab both red corners without also grabbing a blue one.

In [ ]:
xor_table = pd.DataFrame(
    {'x1': [0, 0, 1, 1], 'x2': [0, 1, 0, 1], 'OR-ish': [0, 1, 1, 1], 'AND-ish': [0, 0, 0, 1], 'XOR': [0, 1, 1, 0]}
)
xor_table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8.8, 4.0))
colors = np.where(xor_table['XOR'].to_numpy() == 1, '#EF4444', '#3B82F6')
axes[0].scatter(xor_table['x1'], xor_table['x2'], c=colors, s=120, edgecolor='white', linewidth=1.5)
axes[0].set_title('original x₁,x₂ space')
axes[0].set_xlabel('x₁')
axes[0].set_ylabel('x₂')
axes[0].set_xlim(-0.3, 1.3)
axes[0].set_ylim(-0.3, 1.3)
axes[0].set_aspect('equal')
axes[1].scatter(xor_table['OR-ish'], xor_table['AND-ish'], c=colors, s=120, edgecolor='white', linewidth=1.5)
h = np.linspace(-0.1, 1.2, 50)
axes[1].plot(h, (h - 0.5) / 2, color='#111827', linewidth=2)
axes[1].set_title('new h₁,h₂ space')
axes[1].set_xlabel('h₁ = OR-ish')
axes[1].set_ylabel('h₂ = AND-ish')
axes[1].set_xlim(-0.3, 1.3)
axes[1].set_ylim(-0.3, 1.3)
axes[1].set_aspect('equal')
plt.show()

Look at the right picture: both red rows moved onto `(h1, h2) = (1, 0)`, while the blue
rows are `(0, 0)` and `(1, 1)`. Now `score = OR - 2*AND - 0.5` is positive only for red.

The hidden layer did not bend the output line. It **moved the points into a new space**
where one straight line works. That is Chapter 3 escape route 1, automated.

Work these out on scrap paper, then type your answers in. You'll be told not only
whether you were right, but why the question was worth asking.

In [ ]:
from kidsml import workbook

workbook.render(14)

## 👀 Take a look

Train `[2, 3, 1]` on XOR, then draw each hidden neuron's line and the final boundary.

In [ ]:
X_xor, y_xor = xor_exact()
snaps = mlp_snapshot_training([2, 3, 1], X_xor, y_xor, lr=0.8, epochs=3000, every=150, activation='tanh', seed=2)
model = model_from_snapshot([2, 3, 1], snaps[-1], activation='tanh', seed=2)

fig, ax = plt.subplots(figsize=(5.4, 4.6))
boundary_with_hidden(model, X_xor, y_xor, ax=ax, title='Hidden lines plus final boundary', steps=180)
plt.show()

In [ ]:
hidden_surfaces_figure(model, X_xor, steps=65)

The three coloured lines are the three hidden neurons. Each one sends a different ramp
reading to the output neuron, and the output neuron combines those readings.

Why do they learn different lines? They start with small random differences. After that,
each neuron receives slightly different gradients, so their jobs separate. If every
hidden neuron started identical, they would tend to march in a crowd.

## 🎛️ Your turn

Change the hidden size and activation in the app. Here are three sizes on XOR.

In [ ]:
X_play, y_play = toy_shape('xor', n=180, noise=0.16, seed=3)
fig, axes = plt.subplots(1, 3, figsize=(14, 4.1))
for ax, hidden in zip(axes, [1, 2, 3]):
    m = MLP([2, hidden, 1], activation='tanh', seed=3)
    losses = m.fit(X_play, y_play, lr=0.6, epochs=900, record_every=5)
    decision_boundary(lambda G, model=m: model.predict_proba(G), X_play, y_play, ax=ax, steps=160, title=f'{hidden} hidden')
plt.show()

With one hidden neuron you are mostly back to one learned line. Add a few, and the model
can invent several features before the final neuron decides.

Look for the hidden lines first, then look at the shaded final boundary. The bend appears
in original space because the output line is reading transformed hidden coordinates.

## 💻 In real code

More hidden neurons can help, but they can also wiggle around noisy dots.

In [ ]:
X_over, y_over = toy_shape('moons', n=70, noise=0.32, seed=10)
small = MLP([2, 3, 1], activation='tanh', seed=1)
big = MLP([2, 8, 1], activation='tanh', seed=1)
small.fit(X_over, y_over, lr=0.6, epochs=1200, record_every=20)
big.fit(X_over, y_over, lr=0.6, epochs=2200, record_every=20)
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.4))
decision_boundary(lambda G: small.predict_proba(G), X_over, y_over, ax=axes[0], steps=160, title='3 hidden neurons: calmer')
decision_boundary(lambda G: big.predict_proba(G), X_over, y_over, ax=axes[1], steps=160, title='8 hidden neurons: wobblier')
plt.show()

More hidden neurons give the network more ways to wiggle. That can help with real
patterns, and it can over-study noise. Chapter 15 is about that trade.

## 🏆 Go further

1. **Smallest XOR solver.** What is the fewest hidden neurons that can solve XOR?
2. **Try spiral.** How many hidden neurons does it need before it looks decent?
3. **Set XOR weights by hand.** Use OR-ish and AND-ish to beat training.
4. **Watch the lines.** Scrub training in the app and say what each hidden line learned.
5. 🧸 **Little Kid Corner:** Three friends make a team. Two notice where you stand. The
   last friend listens and decides.

---
**Next up:** Chapter 15 · *Deeper and Wider* — more layers, squishes, and over-studying.